# TinyML Fruit Classification Pipeline (Fruits-360 ➜ Arduino UNO R4)

This notebook is **intentionally verbose and transparent**. Every major block answers:
- **What is happening?**
- **Why are we doing it?**
- **How can you verify correctness?**

The goal is not just accuracy; it is understanding and verification across the full TinyML workflow: data handling, preprocessing, modeling, evaluation, overfitting diagnosis, and embedded deployment constraints.


## 0) Environment & Reproducibility

We set seeds for reproducibility. This does **not** guarantee identical results across hardware/versions, but it reduces randomness.


In [ ]:
import os
import random
import json
import math
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


## 1) Acquire the Fruits-360 dataset (from the official GitHub repository)

**What is happening?**
We download the official Fruits-360 repository ZIP directly from GitHub and extract it locally.

**Why?**
We want a fully reproducible pipeline that does not rely on manual downloads or hard-coded paths.

**How to verify?**
After download and extraction, we will search for the dataset root and confirm that `Training/` and `Test/` directories exist.


In [ ]:
import urllib.request
import zipfile

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

ZIP_URL = "https://github.com/Horea94/Fruit-Images-Dataset/archive/refs/heads/master.zip"
ZIP_PATH = DATA_DIR / "fruit-360.zip"

if not ZIP_PATH.exists():
    print("Downloading Fruits-360 dataset...")
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    print("Download complete:", ZIP_PATH)
else:
    print("Zip already exists:", ZIP_PATH)

EXTRACT_DIR = DATA_DIR / "fruit-360"
if not EXTRACT_DIR.exists():
    EXTRACT_DIR.mkdir()
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Extraction complete.")
else:
    print("Extract directory already exists:", EXTRACT_DIR)


### 1.1) Auto-detect dataset structure (no hard-coded paths)

**What is happening?**
We search inside the extracted directory for a folder that contains both `Training/` and `Test/` directories.

**Why?**
The repository structure can change, and student environments can differ.

**How to verify?**
We print the detected paths and explicitly check for existence.


In [ ]:
def find_dataset_root(search_dir: Path) -> Path:
    candidates = []
    for path in search_dir.rglob("*"):
        if path.is_dir():
            training = path / "Training"
            test = path / "Test"
            if training.is_dir() and test.is_dir():
                candidates.append(path)
    if not candidates:
        raise FileNotFoundError("Could not find dataset root containing Training/ and Test/")
    # If multiple candidates, choose the shallowest
    candidates.sort(key=lambda p: len(p.parts))
    return candidates[0]

DATASET_ROOT = find_dataset_root(EXTRACT_DIR)
TRAIN_DIR = DATASET_ROOT / "Training"
TEST_DIR = DATASET_ROOT / "Test"
OOD_DIR = DATASET_ROOT / "test-multiple_fruits"

print("Dataset root:", DATASET_ROOT)
print("Training exists:", TRAIN_DIR.exists())
print("Test exists:", TEST_DIR.exists())
print("OOD (test-multiple_fruits) exists:", OOD_DIR.exists())


## 2) Choose an 8-class subset transparently

**What is happening?**
We select an **explicit** list of eight classes. If any are missing, we will stop and show the mismatch.

**Why?**
A small subset makes the TinyML model and training process tractable and more interpretable.

**How to verify?**
We list all available classes and show which are selected. We also print class counts for train/test.


In [ ]:
ALL_CLASSES = sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()])
print(f"Total classes found: {len(ALL_CLASSES)}")

SELECTED_CLASSES = [
    "Apple Golden 1",
    "Banana",
    "Cherry 1",
    "Kiwi",
    "Mango",
    "Orange",
    "Peach",
    "Strawberry",
]

missing = [c for c in SELECTED_CLASSES if c not in ALL_CLASSES]
if missing:
    raise ValueError(f"Missing classes in dataset: {missing}")

print("Selected classes:", SELECTED_CLASSES)


In [ ]:
def count_images_per_class(dir_path: Path, classes):
    counts = {}
    for cls in classes:
        cls_dir = dir_path / cls
        if not cls_dir.exists():
            counts[cls] = 0
        else:
            counts[cls] = len([p for p in cls_dir.iterdir() if p.is_file()])
    return counts

train_counts = count_images_per_class(TRAIN_DIR, SELECTED_CLASSES)
test_counts = count_images_per_class(TEST_DIR, SELECTED_CLASSES)

counts_df = pd.DataFrame({"train": train_counts, "test": test_counts})
counts_df["train/test ratio"] = counts_df["train"] / counts_df["test"]
counts_df


**Verification checklist:**
- Each selected class should exist and have non-zero counts.
- Train/test counts should be roughly consistent across classes.


## 3) Raw vs. preprocessed data (transparent inspection)

**What is happening?**
We show **raw images** directly from disk, then show the **preprocessed** images after resizing and normalization.

**Why?**
Students must see the transformation steps and understand that preprocessing changes the input distribution.

**How to verify?**
We sample images randomly (not by batch) to avoid repeated/similar visuals.


In [ ]:
from PIL import Image

IMG_SIZE = (64, 64)

rng = np.random.default_rng(SEED)

# Random unbatched sampling for raw images
raw_samples = []
for cls in SELECTED_CLASSES[:4]:  # show a few classes
    cls_dir = TRAIN_DIR / cls
    img_path = rng.choice([p for p in cls_dir.iterdir() if p.is_file()])
    raw_samples.append((cls, img_path))

plt.figure(figsize=(8, 6))
for i, (cls, path) in enumerate(raw_samples, 1):
    img = Image.open(path)
    plt.subplot(2, 2, i)
    plt.imshow(img)
    plt.title(f"Raw: {cls}")
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Preprocessed view (resize + normalization visualization)
plt.figure(figsize=(8, 6))
for i, (cls, path) in enumerate(raw_samples, 1):
    img = Image.open(path).resize(IMG_SIZE)
    arr = np.asarray(img) / 255.0
    plt.subplot(2, 2, i)
    plt.imshow(arr)
    plt.title(f"Preprocessed: {cls}")
    plt.axis("off")
plt.tight_layout()
plt.show()


**Why not rely on batch visualizations?**
Batch sampling can show repeated or very similar images because batch pipelines may shuffle within a limited buffer. Using unbatched random sampling gives a clearer view of diversity.


## 4) Build TensorFlow datasets

**What is happening?**
We create separate **train** and **test** datasets from the already-defined directory splits. We **do not** mix them.

**Why?**
We want a clean, known split so that evaluation has a clear meaning.

**How to verify?**
We inspect class names, shapes, and sample counts.


In [ ]:
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=SELECTED_CLASSES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=SELECTED_CLASSES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds.class_names
print("Class names:", class_names)

for images, labels in train_ds.take(1):
    print("Batch image shape:", images.shape)
    print("Batch label shape:", labels.shape)


### 4.1) Data augmentation (training only)

**What is happening?**
We apply augmentation **only** to the training set. The test set remains untouched.

**Why?**
Augmentation helps generalization but should not distort evaluation.

**How to verify?**
We visualize augmented images and note their distortions.


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

augmented_train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))


In [ ]:
# Visualize augmented images (note the distortions)
plt.figure(figsize=(8, 6))
for images, labels in augmented_train_ds.take(1):
    for i in range(4):
        plt.subplot(2, 2, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title("Augmented")
        plt.axis("off")
plt.tight_layout()
plt.show()


**Note:** Augmentations can make images look "unnatural". This is expected and is part of learning robustness.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds_prefetch = augmented_train_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds_prefetch = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


## 5) Tiny CNN architecture (Arduino UNO R4 realistic)

**What is happening?**
We build a **tiny** CNN using separable convolutions and global average pooling.

**Why?**
These layers drastically reduce parameter count and computation while preserving accuracy.

**How to verify?**
We print the model summary and parameter count.


In [ ]:
num_classes = len(SELECTED_CLASSES)

model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(*IMG_SIZE, 3)),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.SeparableConv2D(16, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.SeparableConv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.SeparableConv2D(64, 3, activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

param_count = model.count_params()
print(f"Total parameters: {param_count}")


## 6) Train with safeguards against overfitting

**What is happening?**
We train with early stopping and learning-rate scheduling.

**Why?**
To avoid wasting epochs and to reduce overfitting.

**How to verify?**
We plot accuracy and loss for both train and validation/test.


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, monitor='val_loss')
]

history = model.fit(
    train_ds_prefetch,
    validation_data=test_ds_prefetch,
    epochs=30,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
# Plot training curves
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df['accuracy'], label='Train Acc')
plt.plot(history_df['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df['loss'], label='Train Loss')
plt.plot(history_df['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

plt.tight_layout()
plt.show()


**Interpretation guide:**
- If training accuracy climbs while validation accuracy stalls or drops, overfitting is emerging.
- If both are low, underfitting persists.


## 7) Consolidated evaluation (single block)

**What is happening?**
We compute accuracy, precision, recall, F1 (macro & weighted), a per-class report, and a confusion matrix.

**Why?**
Accuracy alone can hide class imbalance or systematic mistakes.

**How to verify?**
We print metrics and visualize a full confusion matrix with all classes.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score

# Get predictions
y_true = []
y_pred = []

for images, labels in test_ds_prefetch:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

acc = accuracy_score(y_true, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

print(f"Accuracy: {acc:.4f}")
print(f"Macro Precision/Recall/F1: {precision_macro:.4f} / {recall_macro:.4f} / {f1_macro:.4f}")
print(f"Weighted Precision/Recall/F1: {precision_weighted:.4f} / {recall_weighted:.4f} / {f1_weighted:.4f}")

print("
Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

# Confusion matrix (always show full matrix)
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha='right')
plt.yticks(np.arange(len(class_names)), class_names)

thresh = cm.max() / 2 if cm.max() > 0 else 1
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j],
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()


## 8) Overfitting & validity checks

Perfect or near-perfect results on Fruits-360 are **suspicious**. The dataset is clean, uniform, and often contains near-duplicates. This can inflate accuracy without real-world robustness.

We run three checks:
1) **Train/Test leakage** via filename and file hash overlap.
2) **Stress-test augmentations** to probe prediction stability.
3) **Out-of-distribution (OOD)** evaluation on `test-multiple_fruits`.


In [ ]:
# 8.1) Leakage check by filename and file hash

def collect_files(dir_path: Path, classes):
    files = []
    for cls in classes:
        cls_dir = dir_path / cls
        files.extend(list(cls_dir.glob("*")))
    return files

train_files = collect_files(TRAIN_DIR, SELECTED_CLASSES)
test_files = collect_files(TEST_DIR, SELECTED_CLASSES)

train_names = set([p.name for p in train_files])
test_names = set([p.name for p in test_files])
name_overlap = train_names.intersection(test_names)

print(f"Filename overlap count: {len(name_overlap)}")

# Hash overlap (MD5) - may take a bit of time

def md5_hash(path: Path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

# Compute hashes
train_hashes = {md5_hash(p) for p in train_files}
test_hashes = {md5_hash(p) for p in test_files}

hash_overlap = train_hashes.intersection(test_hashes)
print(f"Hash overlap count: {len(hash_overlap)}")


In [ ]:
# 8.2) Stress-test augmentation

stress_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.25),
    tf.keras.layers.RandomZoom(0.25),
    tf.keras.layers.RandomContrast(0.3),
])

# Take a small random subset from test set
sample_images = []
sample_labels = []
for images, labels in test_ds.take(3):
    sample_images.append(images)
    sample_labels.append(labels)

sample_images = tf.concat(sample_images, axis=0)
sample_labels = tf.concat(sample_labels, axis=0)

aug_images = stress_aug(sample_images, training=True)

preds_original = model.predict(sample_images, verbose=0)
preds_aug = model.predict(aug_images, verbose=0)

instability = np.mean(np.argmax(preds_original, axis=1) != np.argmax(preds_aug, axis=1))
print(f"Prediction instability under strong augmentation: {instability:.2%}")


In [ ]:
# 8.3) OOD evaluation (test-multiple_fruits)
if OOD_DIR.exists():
    ood_ds = tf.keras.utils.image_dataset_from_directory(
        OOD_DIR,
        labels=None,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    ood_preds = []
    for images in ood_ds:
        preds = model.predict(images, verbose=0)
        ood_preds.extend(np.argmax(preds, axis=1))

    ood_preds = np.array(ood_preds)
    unique, counts = np.unique(ood_preds, return_counts=True)
    dist = {class_names[i]: int(c) for i, c in zip(unique, counts)}
    print("OOD prediction distribution:", json.dumps(dist, indent=2))
else:
    print("OOD directory not found.")


**Interpretation:**
- Any non-zero overlap in hashes indicates **leakage**.
- High instability under strong augmentation suggests the model is brittle.
- OOD predictions often reveal overconfidence on unseen input types.


## 9) INT8 TensorFlow Lite conversion (Arduino-ready)

**What is happening?**
We convert the model to **full INT8** and verify predictions against the original Keras model.

**Why?**
Arduino UNO R4 has limited RAM/flash. INT8 reduces size and speeds inference.

**How to verify?**
We compare predictions on a small test sample between Keras and TFLite.


In [ ]:
# Representative dataset for quantization

def representative_dataset():
    for images, _ in train_ds.take(10):
        yield [tf.cast(images, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

try:
    tflite_model = converter.convert()
    print("TFLite conversion successful. Size (bytes):", len(tflite_model))
except Exception as e:
    raise RuntimeError(f"TFLite conversion failed: {e}")


In [ ]:
# Validate TFLite predictions vs Keras on a small batch

interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

# Grab a small batch
images, labels = next(iter(test_ds.take(1)))

# Keras predictions
keras_preds = model.predict(images, verbose=0)
keras_top = np.argmax(keras_preds, axis=1)

# TFLite predictions
input_scale, input_zero_point = input_details["quantization"]
output_scale, output_zero_point = output_details["quantization"]

# Quantize input
images_int8 = (images / input_scale + input_zero_point).numpy().astype(np.int8)

interpreter.set_tensor(input_details['index'], images_int8)
interpreter.invoke()

output_int8 = interpreter.get_tensor(output_details['index'])
# Dequantize output
output_float = (output_int8.astype(np.float32) - output_zero_point) * output_scale

tflite_top = np.argmax(output_float, axis=1)

agreement = np.mean(keras_top == tflite_top)
print(f"Keras vs TFLite top-1 agreement: {agreement:.2%}")


### 9.1) Export TFLite model for Arduino conversion

The resulting `tflite_model` can be saved and later converted into a C header for Arduino deployment.


In [ ]:
TFLITE_PATH = Path("model_int8.tflite")
TFLITE_PATH.write_bytes(tflite_model)
print("Saved:", TFLITE_PATH)


## 10) Final reflection: Why skepticism matters

Even with strong metrics, Fruits-360 is a **controlled** dataset. Real-world fruits vary in lighting, occlusion, and background.

This notebook demonstrates:
- Transparent data handling
- Deliberate class selection
- Proper augmentation usage
- Model design tailored to TinyML constraints
- Honest evaluation and overfitting checks
- Full INT8 conversion suitable for Arduino UNO R4

Use this as a rigorous starting point, not an endpoint.
